# PPO VM Allocation Experiments

This notebook trains and evaluates PPO agents for VM allocation using the existing DRL pipeline:

- `rl/environment.py`: `VMAllocationEnv` (Gymnasium environment)
- `rl/config.py`: PPO and reward configuration  
- `train_ppo.py`: training script
- `eval_ppo.py`: evaluation script

## Current Config (Updated 2024-12-16) - Balanced Training

| Parameter | Training | Evaluation | Reason |
|-----------|----------|------------|--------|
| `episode_length` | **1440 (12h)** | 17150 (full test) | Half daily cycle |
| `total_timesteps` | **2,000,000** | - | ~1,388 episodes |
| `horizon` | 120 (60min) | 120 | Forecast look-ahead |
| `n_envs` | 8 | - | Parallel training |

**Expected training time**: ~4-5 hours (at 118 it/s)

**Note**: LP vs PPO comparison is done in `lp_vs_ppo_comparison.ipynb`.


In [5]:
# Imports and configuration

from pathlib import Path

# Module imports
import importlib
import train_ppo as train_ppo_module
import eval_ppo as eval_ppo_module
import rl.config as rl_config_module

from rl.config import PPOConfig, SCENARIO_OVERLOAD, SCENARIO_COST
from train_ppo import train_ppo
from eval_ppo import evaluate_scenario, print_comparison

# Reload modules to pick up latest code when notebook stays open
importlib.reload(train_ppo_module)
importlib.reload(eval_ppo_module)
importlib.reload(rl_config_module)

# Refresh config after reload
from rl.config import PPOConfig, SCENARIO_OVERLOAD, SCENARIO_COST

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)

# Show current default PPO configuration
config = PPOConfig()
config


Project root: e:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure


PPOConfig(learning_rate=0.0003, n_steps=2048, batch_size=64, n_epochs=10, gamma=0.99, gae_lambda=0.95, clip_range=0.2, clip_range_vf=None, ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5, episode_length=2880, horizon=120, total_timesteps=5000000, tensorboard_log='./tensorboard_logs/', log_interval=10, save_freq=10000)

In [ ]:
# Configuration (for training or evaluation)
from copy import deepcopy

# Number of parallel envs
N_ENVS = 8

# Create experiment config
exp_config = deepcopy(config)

# ============================================================
# TRAINING CONFIG (Balanced - 12 hours episode)
# ============================================================
# episode_length for TRAINING: 1440 = 12 hours (half daily cycle)
# Balance between learning patterns and training speed
TRAIN_EPISODE_LENGTH = 1440  # 12 hours

# total_timesteps: 2M = ~1,388 episodes with episode_length=1440
exp_config.total_timesteps = 2_000_000

# ============================================================
# EVALUATION CONFIG
# ============================================================
# episode_length for EVALUATION: 17150 = full test set (~6 days)
EVAL_EPISODE_LENGTH = 17150

# Calculate expected metrics
n_episodes = exp_config.total_timesteps // TRAIN_EPISODE_LENGTH
train_hours = TRAIN_EPISODE_LENGTH * 30 / 3600
eval_hours = EVAL_EPISODE_LENGTH * 30 / 3600

print("=" * 60)
print("📋 CONFIGURATION")
print("=" * 60)
print(f"\n🏋️ TRAINING:")
print(f"   • episode_length:   {TRAIN_EPISODE_LENGTH} steps ({train_hours:.0f} hours)")
print(f"   • total_timesteps:  {exp_config.total_timesteps:,}")
print(f"   • expected episodes: ~{n_episodes:,}")
print(f"   • horizon:          {exp_config.horizon} steps")
print(f"   • n_envs:           {N_ENVS}")
print(f"\n📊 EVALUATION:")
print(f"   • episode_length:   {EVAL_EPISODE_LENGTH} steps ({eval_hours:.1f} hours = full test)")
print("=" * 60)


📋 CONFIGURATION

🏋️ TRAINING:
   • episode_length:   2880 steps (24 hours)
   • total_timesteps:  5,000,000
   • expected episodes: ~1,736
   • horizon:          120 steps
   • n_envs:           8

📊 EVALUATION:
   • episode_length:   17150 steps (142.9 hours = full test)


In [7]:
# ============================================================
# TRAINING
# ============================================================
# Set episode_length for training (daily pattern = 2880)
exp_config.episode_length = TRAIN_EPISODE_LENGTH

print("=" * 60)
print("🚀 STARTING TRAINING")
print("=" * 60)
print(f"   • total_timesteps:  {exp_config.total_timesteps:,}")
print(f"   • episode_length:   {exp_config.episode_length} steps (12 hours)")
print(f"   • horizon:          {exp_config.horizon} steps")
print(f"   • n_envs:           {N_ENVS}")
print(f"   • expected time:    ~4-5 hours (with 118 it/s)")
print("=" * 60)

# Train OVERLOAD scenario (Minimize Resource Overload)
print("\n[1/2] Training OVERLOAD scenario...")
model_overload = train_ppo(
    scenario=SCENARIO_OVERLOAD,
    config=exp_config,
    total_timesteps=exp_config.total_timesteps,
    n_envs=N_ENVS,
    continue_training=False,
)

# Train COST scenario (Minimize Operational Cost)
print("\n[2/2] Training COST scenario...")
model_cost = train_ppo(
    scenario=SCENARIO_COST,
    config=exp_config,
    total_timesteps=exp_config.total_timesteps,
    n_envs=N_ENVS,
    continue_training=False,
)

print("\n✅ Training complete for both scenarios!")


🚀 STARTING TRAINING
   • total_timesteps:  5,000,000
   • episode_length:   2880 steps (24 hours)
   • horizon:          120 steps
   • n_envs:           8
   • expected time:    ~2-3 hours

[1/2] Training OVERLOAD scenario...

Training PPO for scenario: OVERLOAD
Latest model: models\random_forest_cpu_total_usage_20251111_221930.pkl
✓ Model loaded: models\random_forest_cpu_total_usage_20251111_221930.pkl
  Model: random_forest
  Target: cpu_total_usage
  Saved at: 2025-11-11 22:19:30
Latest model: models\svr_memory_usage_pct_20251111_184735.pkl
✓ Model loaded: models\svr_memory_usage_pct_20251111_184735.pkl
  Model: svr
  Target: memory_usage_pct
  Saved at: 2025-11-11 18:47:35
Latest model: models\random_forest_system_load_20251111_221944.pkl
✓ Model loaded: models\random_forest_system_load_20251111_221944.pkl
  Model: random_forest
  Target: system_load
  Saved at: 2025-11-11 22:19:44
Creating new PPO model
Using cpu device

Starting training for 5,000,000 timesteps...
Logging to ./t

Output()

------------------------------
| time/              |       |
|    fps             | 104   |
|    iterations      | 1     |
|    time_elapsed    | 156   |
|    total_timesteps | 16384 |
------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.88e+03    |
|    ep_rew_mean          | -8.94e+05   |
| time/                   |             |
|    fps                  | 104         |
|    iterations           | 2           |
|    time_elapsed         | 313         |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.030880727 |
|    clip_fraction        | 0.285       |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.57       |
|    explained_variance   | -1.24       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.147      |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0545     |
|    value_loss           | 0.165       |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.88e+03    |
|    ep_rew_mean          | -8.81e+05   |
| time/                   |             |
|    fps                  | 103         |
|    iterations           | 3           |
|    time_elapsed         | 473         |
|    total_timesteps      | 49152       |
| train/                  |             |
|    approx_kl            | 0.031804968 |
|    clip_fraction        | 0.325       |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.54       |
|    explained_variance   | -0.191      |
|    learning_rate        | 0.0003      |
|    loss                 | -0.175      |
|    n_updates            | 20          |
|    policy_gradient_loss | -0.0667     |
|    value_loss           | 0.0343      |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.88e+03    |
|    ep_rew_mean          | -8.81e+05   |
| time/                   |             |
|    fps                  | 103         |
|    iterations           | 4           |
|    time_elapsed         | 632         |
|    total_timesteps      | 65536       |
| train/                  |             |
|    approx_kl            | 0.032701798 |
|    clip_fraction        | 0.349       |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.5        |
|    explained_variance   | 0.421       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.18       |
|    n_updates            | 30          |
|    policy_gradient_loss | -0.0662     |
|    value_loss           | 0.0204      |
-----------------------------------------


Eval num_timesteps=80000, episode_reward=-344846.66 +/- 0.00

Episode length: 2880.00 +/- 0.00

-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 2.88e+03    |
|    mean_reward          | -3.45e+05   |
| time/                   |             |
|    total_timesteps      | 80000       |
| train/                  |             |
|    approx_kl            | 0.037370913 |
|    clip_fraction        | 0.391       |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.45       |
|    explained_variance   | 0.366       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.172      |
|    n_updates            | 40          |
|    policy_gradient_loss | -0.0746     |
|    value_loss           | 0.0151      |
-----------------------------------------


New best mean reward!

In [ ]:
# ============================================================
# EVALUATION (Full Test Set)
# ============================================================
print("=" * 60)
print("📊 EVALUATION ON FULL TEST SET")
print("=" * 60)
print(f"   • episode_length: {EVAL_EPISODE_LENGTH} steps (full test)")
print("=" * 60)

results = {}

for scenario in [SCENARIO_OVERLOAD, SCENARIO_COST]:
    print(f"\n🔍 Evaluating {scenario.upper()} scenario...")
    comp = evaluate_scenario(
        scenario,
        episode_length=EVAL_EPISODE_LENGTH,  # Full test set
        horizon=exp_config.horizon,
    )
    results[scenario] = comp
    print_comparison(comp)

print("\n✅ Evaluation complete!")
print("📁 Results saved to: results/ppo_schedule_test_*.csv")
results



Evaluating scenario: OVERLOAD
Latest model: models\random_forest_cpu_total_usage_20251111_221930.pkl
✓ Model loaded: models\random_forest_cpu_total_usage_20251111_221930.pkl
  Model: random_forest
  Target: cpu_total_usage
  Saved at: 2025-11-11 22:19:30
Latest model: models\svr_memory_usage_pct_20251111_184735.pkl
✓ Model loaded: models\svr_memory_usage_pct_20251111_184735.pkl
  Model: svr
  Target: memory_usage_pct
  Saved at: 2025-11-11 18:47:35
Latest model: models\random_forest_system_load_20251111_221944.pkl
✓ Model loaded: models\random_forest_system_load_20251111_221944.pkl
  Model: random_forest
  Target: system_load
  Saved at: 2025-11-11 22:19:44
Loaded model from E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\rl_models\ppo_overload.zip

Running PPO rollout for scenario: overload
Latest model: models\random_forest_cpu_total_usage_20251111_221930.pkl
✓ Model loaded: models\random_forest_cpu_total_usage_20251111_221930.pkl
  Model: random_for

{'overload': {'scenario': 'overload',
  'timestamp': '2025-12-16 23:45:32',
  'ppo': {'total_vm_cost': 2784.5056,
   'total_switching_cost': 119.75000000000001,
   'total_cost': 2904.2556000000004,
   'sla_violations': 0,
   'sla_violation_rate': 0.0,
   'mean_cpu_utilization': 0.02776361366262689,
   'mean_mem_utilization': 0.0,
   'n_steps': 17150},
  'ppo_30min': {'total_vm_cost': 0.0,
   'total_switching_cost': 0.0,
   'total_cost': 0.0,
   'sla_violations': 0,
   'sla_violation_rate': 0.0,
   'n_buckets': 287}},
 'cost': {'scenario': 'cost',
  'timestamp': '2025-12-16 23:47:30',
  'ppo': {'total_vm_cost': 291.2416,
   'total_switching_cost': 35.400000000000006,
   'total_cost': 326.64160000000004,
   'sla_violations': 40,
   'sla_violation_rate': 0.0023323615160349854,
   'mean_cpu_utilization': 0.37813913267239807,
   'mean_mem_utilization': 0.0,
   'n_steps': 17150},
  'ppo_30min': {'total_vm_cost': 0.0,
   'total_switching_cost': 0.0,
   'total_cost': 0.0,
   'sla_violations': 

In [ ]:
# Inspect generated PPO schedules (PER-STEP results)

import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("forecast_result")

# Load PPO per-step schedules
ppo_overload_path = RESULTS_DIR / "ppo_schedule_test_overload.csv"
ppo_cost_path = RESULTS_DIR / "ppo_schedule_test_cost.csv"

print("=== PPO Per-Step Schedules ===")
print(f"PPO overload: {ppo_overload_path.exists()}")
print(f"PPO cost: {ppo_cost_path.exists()}")

ppo_overload_df = pd.read_csv(ppo_overload_path) if ppo_overload_path.exists() else None
ppo_cost_df = pd.read_csv(ppo_cost_path) if ppo_cost_path.exists() else None

if ppo_overload_df is not None:
    print(f"\nPPO Overload: {len(ppo_overload_df)} steps")
    display(ppo_overload_df.head(10))

if ppo_cost_df is not None:
    print(f"\nPPO Cost: {len(ppo_cost_df)} steps")
    display(ppo_cost_df.head(10))


=== PPO Per-Step Schedules ===
PPO overload: True
PPO cost: True

PPO Overload: 17150 steps


,timestamp,allocation,vm_cost_per_hour,switching_cost,total_cost_per_hour,cpu_allocated_cores,mem_allocated_gb,cpu_vm_only,mem_vm_only,cpu_required_cores,mem_required_gb,cpu_overflow_cores,mem_overflow_gb,cpu_utilization_pct,mem_utilization_pct,sla_violation_flag
0,1970-01-25 01:05:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.24,0.258175,0.0,0,0.0,0.0,0
1,1970-01-25 01:06:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.22,0.258186,0.0,0,0.0,0.0,0
2,1970-01-25 01:06:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.13,0.258044,0.0,0,0.0,0.0,0
3,1970-01-25 01:07:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.08,0.257840,0.0,0,0.0,0.0,0
4,1970-01-25 01:07:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.10,0.258327,0.0,0,0.0,0.0,0
5,1970-01-25 01:08:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.23,0.258050,0.0,0,0.0,0.0,0
6,1970-01-25 01:08:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.14,0.258136,0.0,0,0.0,0.0,0
7,1970-01-25 01:09:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.21,0.258079,0.0,0,0.0,0.0,0
8,1970-01-25 01:09:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.12,0.258266,0.0,0,0.0,0.0,0
9,1970-01-25 01:10:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.07,0.258260,0.0,0,0.0,0.0,0



PPO Cost: 17150 steps


,timestamp,allocation,vm_cost_per_hour,switching_cost,total_cost_per_hour,cpu_allocated_cores,mem_allocated_gb,cpu_vm_only,mem_vm_only,cpu_required_cores,mem_required_gb,cpu_overflow_cores,mem_overflow_gb,cpu_utilization_pct,mem_utilization_pct,sla_violation_flag
0,1970-01-25 01:05:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.24,0.258175,0.0,0,0.0,0.0,0
1,1970-01-25 01:06:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.22,0.258186,0.0,0,0.0,0.0,0
2,1970-01-25 01:06:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.13,0.258044,0.0,0,0.0,0.0,0
3,1970-01-25 01:07:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.08,0.257840,0.0,0,0.0,0.0,0
4,1970-01-25 01:07:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.10,0.258327,0.0,0,0.0,0.0,0
5,1970-01-25 01:08:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.23,0.258050,0.0,0,0.0,0.0,0
6,1970-01-25 01:08:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.14,0.258136,0.0,0,0.0,0.0,0
7,1970-01-25 01:09:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.21,0.258079,0.0,0,0.0,0.0,0
8,1970-01-25 01:09:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.12,0.258266,0.0,0,0.0,0.0,0
9,1970-01-25 01:10:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.07,0.258260,0.0,0,0.0,0.0,0
